# VORTEX-AI (CG-NSDE) - Google Colab Training

**Parallel training on Colab T4 GPU**

**Prerequisites:**
- Kaggle account with API token (username + key)
- Access to `brxkeaaryan/vortex-ai-data` dataset (374 MB)
- GitHub repo access

**Outputs:** Checkpoints in `/content/vortex-ai/models/checkpoints/`, logs in `/content/vortex-ai/runs/`

## 1. Environment & GPU Check

In [ ]:
import os, sys, subprocess, shutil
import torch

print("python", sys.version.split()[0])
print("torch", torch.__version__, "| cuda", torch.version.cuda)
print("gpus", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(" ", i, torch.cuda.get_device_name(i))

## 2. Kaggle Authentication (Use API Token from Colab Secrets)

In [ ]:
# Option 1: Use Colab Secret (recommended)
# In Colab: left sidebar 🔑 → Add secrets named KAGGLE_USERNAME and KAGGLE_KEY
from google.colab import userdata
import os, json

try:
    kaggle_username = userdata.get('KAGGLE_USERNAME')
    kaggle_key = userdata.get('KAGGLE_KEY')
    config = {"username": kaggle_username, "key": kaggle_key}
    print("✓ Using KAGGLE_USERNAME/KAGGLE_KEY from Colab Secrets")
except:
    # Option 2: Fallback to manual entry
    kaggle_username = input('Enter Kaggle username: ').strip()
    import getpass
    kaggle_key = getpass.getpass('Enter Kaggle API key: ').strip()
    config = {"username": kaggle_username, "key": kaggle_key}
    print("✓ Using manually entered credentials")

# Write proper kaggle.json format
!mkdir -p ~/.kaggle
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump(config, f)
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

# Verify access to data
!kaggle datasets files brxkeaaryan/vortex-ai-data

## 3. Clone Repository & Install Dependencies

In [ ]:
REPO_URL = "https://github.com/yash0238/Vortex-AI.git"  # Update if forked
WORK_DIR = "/content/vortex-ai"

# Change to /content first to avoid shell-init error when deleting WORK_DIR
os.chdir("/content")
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

!git clone {REPO_URL} {WORK_DIR}
os.chdir(WORK_DIR)
print("workdir:", os.getcwd())

In [ ]:
# Install dependencies (Colab has torch+cu118; install extras)
REQUIRED = {
    "torch_geometric": "torch-geometric",
    "torchsde": "torchsde",
    "pytorch_lightning": "pytorch-lightning",
    "wandb": "wandb",
}

missing = []
for module, package in REQUIRED.items():
    try:
        __import__(module)
    except ImportError:
        missing.append(package)

if missing:
    print("installing:", missing, flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("all dependencies already present")

import pytorch_lightning
import torch_geometric
import torchsde

print("torch_geometric", torch_geometric.__version__)
print("torchsde", torchsde.__version__)
print("pytorch_lightning", pytorch_lightning.__version__)

## 4. Download Data Artifacts from Kaggle (~374 MB)

In [ ]:
DATA_DIR = os.path.join(WORK_DIR, "data", "raw")
os.makedirs(DATA_DIR, exist_ok=True)

# Download dataset (374 MB) - this takes 1-2 minutes
!kaggle datasets download brxkeaaryan/vortex-ai-data -p {DATA_DIR} --unzip

# Verify
import numpy as np
windows = np.load(os.path.join(DATA_DIR, "windows.npy"))
adj = np.load(os.path.join(DATA_DIR, "adj_matrices.npy"))
feats = np.load(os.path.join(DATA_DIR, "window_node_features.npy"))

print("windows     ", windows.shape)
print("adj_matrices", adj.shape)
print("node_feats  ", feats.shape)

n, T, N = windows.shape
assert (T, N) == (60, 50), f"unexpected window shape {windows.shape}"
assert adj.shape == (n, N, N), f"unexpected adjacency shape {adj.shape}"
assert feats.shape[1:] == (T, N, 6), f"unexpected node-feature shape {feats.shape}"
assert feats.shape[0] in (n, n - 1), f"node-feature length {feats.shape[0]} vs {n} windows"

for name in ("scaler.npz", "split.npz", "window_regimes_v2.npy", "volume_windows.npy", "nifty50_close.csv"):
    assert os.path.exists(os.path.join(DATA_DIR, name)), f"missing artifact: {name}"

print(f"OK: {n} windows x {T} days x {N} stocks")

## 5. Training Configuration

In [ ]:
# Config - adjust for smoke test or full run
SMOKE = False          # True -> 2-epoch validation run
MAX_EPOCHS = 2 if SMOKE else 200
BATCH_SIZE = 4
NUM_WORKERS = 2
RUN_TAG = "colab_smoke" if SMOKE else "colab_t4"

# Optional: Enable wandb (set your API key)
# os.environ["WANDB_API_KEY"] = "your_key_here"

TRAIN_SCRIPT = os.path.join(WORK_DIR, "train_vortex.py")
CONFIG_PATH = os.path.join(WORK_DIR, "training", "config.yaml")

cmd = [
    "python", TRAIN_SCRIPT,
    "--config", CONFIG_PATH,
    "--accelerator", "cuda",
    "--devices", "1",
    "--max_epochs", str(MAX_EPOCHS),
    "--batch_size", str(BATCH_SIZE),
    "--num_workers", str(NUM_WORKERS),
    "--run_tag", RUN_TAG,
]

print("$ " + " ".join(cmd), flush=True)

## 6. Run Training

In [ ]:
# Run training with GPU auto-detection + live epoch logging
import subprocess, time, sys, torch

# Auto-detect GPU
ACCELERATOR = "cuda" if torch.cuda.is_available() else "cpu"
DEVICES = 1 if torch.cuda.is_available() else "auto"

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  GPU NOT ENABLED - Go to Runtime → Change runtime type → T4 GPU → Save")

# Override config accelerator/devices via CLI
TRAIN_SCRIPT = os.path.join(WORK_DIR, "train_vortex.py")
CONFIG_PATH = os.path.join(WORK_DIR, "training", "config.yaml")

cmd = [
    sys.executable, TRAIN_SCRIPT,
    "--config", CONFIG_PATH,
    "--accelerator", ACCELERATOR,
    "--devices", str(DEVICES),
    "--max_epochs", str(MAX_EPOCHS),
    "--batch_size", str(BATCH_SIZE),
    "--num_workers", str(NUM_WORKERS),
    "--run_tag", RUN_TAG,
]

print(f"\n{'='*60}")
print(f"Starting training: accelerator={ACCELERATOR}, devices={DEVICES}")
print(f"Command: {' '.join(cmd)}")
print(f"{'='*60}\n")

# Run with real-time output streaming
start = time.time()
process = subprocess.Popen(
    cmd,
    cwd=WORK_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,  # Combine stderr into stdout
    text=True,
    bufsize=1,  # Line buffered
)

# Stream output line by line
for line in process.stdout:
    print(line, end='')  # Real-time epoch logs

process.wait()
elapsed = time.time() - start

if process.returncode != 0:
    print(f"\n❌ Training failed (exit code: {process.returncode})")
else:
    print(f"\n✅ Training completed in {elapsed/60:.1f} minutes")

## 7. Evaluate Best Checkpoint (Skipped in Smoke Mode)

In [ ]:
if SMOKE:
    print("smoke run: skipping evaluation")
else:
    import glob
    ckpts = glob.glob(os.path.join(WORK_DIR, "models", "checkpoints", "vortex-epoch=*.ckpt"))
    if ckpts:
        ckpts.sort(key=lambda x: int(x.split("epoch=")[1].split("-")[0]))
        best_ckpt = ckpts[-1]
        print(f"Best checkpoint: {best_ckpt}")
        
        eval_cmd = [
            "python",
            os.path.join(WORK_DIR, "evaluate_vortex.py"),
            "--checkpoint", best_ckpt,
            "--n_scenarios", "500"
        ]
        subprocess.run(eval_cmd, check=True, cwd=WORK_DIR)
    else:
        print("No checkpoints found")

## 8. Download Results

In [ ]:
# Zip results for download
!cd /content/vortex-ai && zip -r results.zip models/checkpoints runs

# Download via Colab UI
from google.colab import files
files.download("/content/vortex-ai/results.zip")